In [1]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/ngocccchannn/raw-embeddings/raw_course_text_embeddings.npy
/kaggle/input/datasets/ngocccchannn/raw-embeddings/raw_embeddings_course_index.json
/kaggle/input/datasets/ngocccchannn/cold-scenario-data/test_cold.parquet
/kaggle/input/datasets/ngocccchannn/cold-scenario-data/train_cold.parquet
/kaggle/input/datasets/ngocccchannn/cold-scenario-data/val_cold.parquet
/kaggle/input/datasets/ngocccchannn/evaluator/evaluator.py
/kaggle/input/datasets/ngocccchannn/cold-scenario/cold_items_test.json
/kaggle/input/datasets/ngocccchannn/cold-scenario/cold_items_val.json


In [2]:
import os
import sys
import copy
import json
import random
import hashlib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import subprocess

def set_seed(s=42):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)
set_seed(42)
print("Đã cố định random seed!")

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Sử dụng thiết bị: {DEVICE}")

sys.path.append('/kaggle/input/datasets/ngocccchannn/evaluator') 
from evaluator import evaluate

Đã cố định random seed!
Sử dụng thiết bị: cuda


In [3]:
print("Đang tải dữ liệu...")
train_df = pd.read_parquet('/kaggle/input/datasets/ngocccchannn/cold-scenario-data/train_cold.parquet')

with open('/kaggle/input/datasets/ngocccchannn/raw-embeddings/raw_embeddings_course_index.json', 'r') as f:
    course_ids = json.load(f)['course_id']

assert len(course_ids) == 3769, f"Bảng chỉ số phải có đúng 3769 khóa, nhận {len(course_ids)}"
item2idx = {str(c): i for i, c in enumerate(course_ids)}

users = sorted(train_df['user_id'].unique())
user2idx = {str(u): i for i, u in enumerate(users)}

num_users = len(user2idx)
num_items = len(item2idx)
print(f"Tổng số User: {num_users}, Tổng số Item: {num_items}")

Đang tải dữ liệu...
Tổng số User: 98220, Tổng số Item: 3769


In [4]:
with open('/kaggle/input/datasets/ngocccchannn/cold-scenario/cold_items_val.json', 'r') as f:
    cold_val_ids = json.load(f)

with open('/kaggle/input/datasets/ngocccchannn/cold-scenario/cold_items_test.json', 'r') as f:
    cold_test_ids = json.load(f)

cold_val_ids = cold_val_ids.get("cold_items", cold_val_ids) if isinstance(cold_val_ids, dict) else cold_val_ids
cold_test_ids = cold_test_ids.get("cold_items", cold_test_ids) if isinstance(cold_test_ids, dict) else cold_test_ids

cold_item_set = set(str(c) for c in cold_val_ids + cold_test_ids)
cold_idx = torch.tensor(sorted(item2idx[c] for c in cold_item_set if c in item2idx), dtype=torch.long)

# Chuyển tương tác sang index
u_all = train_df['user_id'].map(lambda x: user2idx[str(x)]).to_numpy()
i_all = train_df['course_id'].map(lambda x: item2idx[str(x)]).to_numpy()

warm_idx = torch.tensor(np.unique(i_all), dtype=torch.long) # Khóa có trong train

print(f"Tổng số User train: {num_users}, Tổng số Item hệ thống: {num_items}")
print(f"Số lượng khóa Warm trong train: {len(warm_idx)}, Số lượng khóa Lạnh (Val+Test): {len(cold_idx)}")

Tổng số User train: 98220, Tổng số Item hệ thống: 3769
Số lượng khóa Warm trong train: 2401, Số lượng khóa Lạnh (Val+Test): 753


In [5]:
# Tách kiểm định nội bộ: Mỗi học viên (>= 2 tương tác) chọn ngẫu nhiên 1 tương tác (tổng ~5% train)
cnt = train_df.groupby("user_id")["course_id"].transform("size")
one = train_df[cnt >= 2].groupby("user_id").sample(1, random_state=42)
val_rows = one.sample(n=min(len(one), int(0.05 * len(train_df))), random_state=42).index
is_val = train_df.index.isin(val_rows)

# Tương tác dùng để huấn luyện đồ thị (loại bỏ các dòng val)
train_mask = ~is_val
internal_train_u = torch.tensor(u_all[train_mask], dtype=torch.long)
internal_train_i = torch.tensor(i_all[train_mask], dtype=torch.long)

# Tương tác làm ground-truth cho validation nội bộ
val_int_u = u_all[is_val]
val_int_i = i_all[is_val]

val_target_dict = {}
for u, i in zip(val_int_u, val_int_i):
    u = int(u)
    if u not in val_target_dict:
        val_target_dict[u] = []
    val_target_dict[u].append(int(i))

# Mặt nạ loại bỏ các tương tác huấn luyện của học viên khi đánh giá nội bộ
train_hist_dict = {}
for u, i in zip(internal_train_u.numpy(), internal_train_i.numpy()):
    u = int(u)
    if u not in train_hist_dict:
        train_hist_dict[u] = set()
    train_hist_dict[u].add(int(i))

# Xây dựng ma trận kề chuẩn hóa D^{-1/2} A D^{-1/2} cho đồ thị huấn luyện (~is_val)
print("Đang xây dựng ma trận kề chuẩn hóa cho đồ thị huấn luyện...")
row = torch.cat([internal_train_u, internal_train_i + num_users])
col = torch.cat([internal_train_i + num_users, internal_train_u])
val = torch.ones(row.size(0), dtype=torch.float32)

assoc = torch.sparse_coo_tensor(torch.stack([row, col]), val, (num_users + num_items, num_users + num_items)).to(DEVICE)
deg = torch.sparse.sum(assoc, dim=1).to_dense()

deg_inv_sqrt = torch.pow(deg, -0.5)
deg_inv_sqrt[deg_inv_sqrt == float('inf')] = 0.0

i_idx = torch.arange(num_users + num_items, device=DEVICE)
diag_indices = torch.stack([i_idx, i_idx], dim=0)

deg_mat_inv_sqrt = torch.sparse_coo_tensor(
    diag_indices, 
    deg_inv_sqrt.to(DEVICE), 
    (num_users + num_items, num_users + num_items)
)

adj_mat = torch.sparse.mm(deg_mat_inv_sqrt, torch.sparse.mm(assoc, deg_mat_inv_sqrt))

# Xây dựng ma trận kề đầy đủ (Full Graph) để tính embedding cuối cùng sau khi hoàn tất huấn luyện
row_full = torch.cat([torch.tensor(u_all, dtype=torch.long), torch.tensor(i_all, dtype=torch.long) + num_users])
col_full = torch.cat([torch.tensor(i_all, dtype=torch.long) + num_users, torch.tensor(u_all, dtype=torch.long)])
val_full = torch.ones(row_full.size(0), dtype=torch.float32)

assoc_full = torch.sparse_coo_tensor(torch.stack([row_full, col_full]), val_full, (num_users + num_items, num_users + num_items)).to(DEVICE)
deg_full = torch.sparse.sum(assoc_full, dim=1).to_dense()
deg_inv_sqrt_full = torch.pow(deg_full, -0.5)
deg_inv_sqrt_full[deg_inv_sqrt_full == float('inf')] = 0.0

deg_mat_inv_sqrt_full = torch.sparse_coo_tensor(
    diag_indices, 
    deg_inv_sqrt_full.to(DEVICE), 
    (num_users + num_items, num_users + num_items)
)

adj_mat_full = torch.sparse.mm(deg_mat_inv_sqrt_full, torch.sparse.mm(assoc_full, deg_mat_inv_sqrt_full))

Đang xây dựng ma trận kề chuẩn hóa cho đồ thị huấn luyện...


/tmp/ipykernel_23/1122797195.py:37: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:760.)
  assoc = torch.sparse_coo_tensor(torch.stack([row, col]), val, (num_users + num_items, num_users + num_items)).to(DEVICE)


In [6]:
class LightGCN(nn.Module):
    def __init__(self, num_users, num_items, latent_dim=64, n_layers=3):
        super(LightGCN, self).__init__()
        self.num_users = num_users
        self.num_items = num_items
        self.latent_dim = latent_dim
        self.n_layers = n_layers
        
        self.user_embedding = nn.Embedding(num_users, latent_dim)
        self.item_embedding = nn.Embedding(num_items, latent_dim)
        nn.init.xavier_uniform_(self.user_embedding.weight)
        nn.init.xavier_uniform_(self.item_embedding.weight)

    def forward(self, adj_matrix): 
        users_emb = self.user_embedding.weight
        items_emb = self.item_embedding.weight
        all_emb = torch.cat([users_emb, items_emb], dim=0)
        
        embs = [all_emb]
        for _ in range(self.n_layers):
            all_emb = torch.sparse.mm(adj_matrix, all_emb)
            embs.append(all_emb)
            
        lightgcn_emb = torch.stack(embs, dim=1).mean(dim=1)
        users_emb_final, items_emb_final = torch.split(lightgcn_emb, [self.num_users, self.num_items])
        return users_emb_final, items_emb_final

In [7]:
LATENT_DIM = 64
N_LAYERS = 3
BATCH_SIZE = 1024
LR = 1e-3
EPOCHS = 60

model = LightGCN(num_users, num_items, latent_dim=LATENT_DIM, n_layers=N_LAYERS).to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=LR)

def bpr_loss(users, pos_items, neg_items, u_emb, i_emb):
    u_g = u_emb[users]
    pos_g = i_emb[pos_items]
    neg_g = i_emb[neg_items]
    pos_scores = torch.sum(u_g * pos_g, dim=1)
    neg_scores = torch.sum(u_g * neg_g, dim=1)
    return -torch.mean(torch.nn.functional.logsigmoid(pos_scores - neg_scores))

print("--- Bắt đầu huấn luyện LightGCN nền chung ---")
best_recall = 0.0
best_state = None

warm_idx_device = warm_idx.to(DEVICE)

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    
    perm = torch.randperm(len(internal_train_u))
    sampled_u = internal_train_u[perm]
    sampled_pos_i = internal_train_i[perm]
    
    # Mẫu âm CHỈ lấy trong các khóa ấm (warm_idx) theo Phụ lục A.1
    rand_indices = torch.randint(0, len(warm_idx), (len(perm),), dtype=torch.long)
    sampled_neg_i = warm_idx[rand_indices]
    
    for s in range(0, len(perm), BATCH_SIZE):
        batch_u = sampled_u[s:s+BATCH_SIZE].to(DEVICE)
        batch_pos = sampled_pos_i[s:s+BATCH_SIZE].to(DEVICE)
        batch_neg = sampled_neg_i[s:s+BATCH_SIZE].to(DEVICE)
        
        optimizer.zero_grad()
        u_emb, i_emb = model(adj_mat)
        loss = bpr_loss(batch_u, batch_pos, batch_neg, u_emb, i_emb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        
    # Đánh giá Early Stopping trên tập validation nội bộ
    model.eval()
    with torch.no_grad():
        u_emb, i_emb = model(adj_mat)
        
        def score_fn(users_tensor):
            u_idx = users_tensor.to(DEVICE)
            return torch.matmul(u_emb[u_idx], i_emb.t())
        
        val_res = evaluate(score_fn, val_target_dict, mask=train_hist_dict, ks=(20,), bs=1024)
        current_recall = val_res['Recall@20'].mean()
        
    print(f"Epoch {epoch+1:02d}/{EPOCHS} | Loss: {total_loss:.4f} | Recall@20 (Val): {current_recall:.4f}")
    
    # Dùng deepcopy để sao chép trọng số tốt nhất thực sự theo Phụ lục A.1
    if current_recall > best_recall:
        best_recall = current_recall
        best_state = copy.deepcopy(model.state_dict())

print(f"\nHuấn luyện hoàn thành! Best Recall@20: {best_recall:.4f}")

--- Bắt đầu huấn luyện LightGCN nền chung ---
Epoch 01/60 | Loss: 135.8613 | Recall@20 (Val): 0.2229
Epoch 02/60 | Loss: 89.7535 | Recall@20 (Val): 0.2393
Epoch 03/60 | Loss: 78.1521 | Recall@20 (Val): 0.2615
Epoch 04/60 | Loss: 70.4074 | Recall@20 (Val): 0.2840
Epoch 05/60 | Loss: 63.9617 | Recall@20 (Val): 0.3056
Epoch 06/60 | Loss: 59.0204 | Recall@20 (Val): 0.3206
Epoch 07/60 | Loss: 55.3822 | Recall@20 (Val): 0.3287
Epoch 08/60 | Loss: 52.5739 | Recall@20 (Val): 0.3360
Epoch 09/60 | Loss: 49.8026 | Recall@20 (Val): 0.3412
Epoch 10/60 | Loss: 47.3721 | Recall@20 (Val): 0.3507
Epoch 11/60 | Loss: 45.2221 | Recall@20 (Val): 0.3560
Epoch 12/60 | Loss: 43.2679 | Recall@20 (Val): 0.3619
Epoch 13/60 | Loss: 41.8118 | Recall@20 (Val): 0.3684
Epoch 14/60 | Loss: 40.2817 | Recall@20 (Val): 0.3719
Epoch 15/60 | Loss: 39.4659 | Recall@20 (Val): 0.3782
Epoch 16/60 | Loss: 37.5501 | Recall@20 (Val): 0.3805
Epoch 17/60 | Loss: 36.4895 | Recall@20 (Val): 0.3852
Epoch 18/60 | Loss: 35.8176 | Recal

In [8]:
# 1. Nạp trọng số tốt nhất thu được từ quá trình dừng sớm
model.load_state_dict(best_state)
model.eval()

# 2. Tính toán embedding trên đồ thị đầy đủ (gồm cả các dòng kiểm định)
with torch.no_grad():
    u_emb_final, i_emb_final = model(adj_mat_full)
    
    g_u = u_emb_final.cpu().numpy()
    g_i = i_emb_final.cpu().numpy()

# 3. Gán g_i = 0.0 cho tất cả khóa lạnh theo Phụ lục A.1
g_i[cold_idx.numpy()] = 0.0

# 4. Kiểm tra kích thước hình dạng đúng chuẩn (3.769 khóa)
assert g_u.shape == (num_users, 64), f"Lỗi shape g_u: {g_u.shape}"
assert g_i.shape == (3769, 64), f"Lỗi shape g_i: {g_i.shape}"
assert np.allclose(g_i[cold_idx.numpy()], 0.0), "Lỗi: Khóa lạnh chưa được gán về 0!"

# 5. Lưu ma trận embedding nền chung
np.save("g_u.npy", g_u)
np.save("g_i.npy", g_i)

# 6. Lưu file index phục vụ assert cho các notebook sau theo Phụ lục A.3
with open("item_index.json", "w") as f:
    json.dump(course_ids, f)

with open("user_index_cold.json", "w") as f:
    json.dump(users, f)

# 7. In mã checksum MD5 xác nhận tính toàn vẹn
md5_res = {p: hashlib.md5(open(p, "rb").read()).hexdigest() for p in ("g_u.npy", "g_i.npy")}
print("Xuất file hoàn tất thành công!")
print(f"Bảng kích thước: g_u = {g_u.shape}, g_i = {g_i.shape}")
print(f"MD5 Checksum: {md5_res}")

Xuất file hoàn tất thành công!
Bảng kích thước: g_u = (98220, 64), g_i = (3769, 64)
MD5 Checksum: {'g_u.npy': '802c799ac3a99970be876a005a494a21', 'g_i.npy': '23fe413552cabc075c28ad581e65361f'}
